<h1>Week 2: Foundations of Network Security </h1>

<h2>Step 1: Key Generation</h2>

In [ ]:
# Import required cryptography libraries
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import rsa
# Generate a 2048-bit RSA private key
private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)

# Opens a file called private eky and writes it in binary which is than saved in a PEM format
# and it is saved without a password.
with open("private_key.pem", "wb") as f:
    f.write(
        private_key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
        )
)
# Derive the public key from the private key
public_key = private_key.public_key()
# Save the public key to a file in PEM format
with open("public_key.pem", "wb") as f:
    f.write(
        public_key.public_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PublicFormat.SubjectPublicKeyInfo
        )
    )
print("Keys generated and saved successfully: private_key.pem, public_key.pem")

✅ Keys saved: private_key.pem, public_key.pem


<h2>Step 2: Receiver (Server) – Receive & Decrypt</h2>

In [ ]:
# Libraries that are being used and imported
import socket
import pickle
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
# Loads the receivers keys to dectypt AES key.
# The AES key is apart of a symmetric encryption method that uses same key for encrypting
# and decrypting.
with open("private_key.pem", "rb") as f:
    private_key = serialization.load_pem_private_key(f.read(), password=None)

# Creates a TCP socket connection that binds to port 65432 and it is in listening mode.
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.bind(("localhost", 65432))
    s.listen()
    print(" Waiting for connection...")
    # Blocks until sender connects
    conn, addr = s.accept() 
    # Connects and prepares a byte string to store the data
    with conn:
        print(f" Connected by {addr}")
        data = b""
        # Receives the data in chuns which are than added together
        while True:
            chunk = conn.recv(4096)
            if not chunk:
                break   
            data += chunk
# Unpacks the recieved data and bundles it using the pickle function.
# IV = AES initialisation variable. 
encrypted_key, iv, encrypted_message = pickle.loads(data)
# Uses RSA to decrypt AES key, the decrypted AES key holds the real key values.
aes_key = private_key.decrypt(
    encrypted_key,
    padding.OAEP(
    mgf=padding.MGF1(algorithm=hashes.SHA256()),
    algorithm=hashes.SHA256(),
    label=None
    )
)
# Creates AES cipher and performs the actual decryption
cipher = Cipher(algorithms.AES(aes_key), modes.CFB(iv))
decryptor = cipher.decryptor()
# Decrypts and prints the originial message
message = decryptor.update(encrypted_message) + decryptor.finalize()
print(" Decrypted message:", message.decode())

 Waiting for connection...
 Connected by ('127.0.0.1', 32254)
 Decrypted message: Hello from the secure sender! This is confidential.


<h2>Step 3: Sender (Client) – Encrypt & Send</h2>

Note: The code for this can be found in Network_Security\Lab_2\sender.ipynb. This code is being pasted just for presentation.

In [ ]:
# Imports libaries and modules
import sys
!{sys.executable} -m pip install cryptography
import socket
import os
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
import pickle
# Loads the receivers keyand reads it and creates a message to be sent
with open("public_key.pem", "rb") as f:
    public_key = serialization.load_pem_public_key(f.read())
message = b"Secure message from sender: this data is confidential."
# Random 32 byte aes key
aes_key = os.urandom(32) 
# Random vector to ensure unique ciphertext
iv = os.urandom(16)
# Creayes an AES cipher object
cipher = Cipher(algorithms.AES(aes_key), modes.CFB(iv))
encryptor = cipher.encryptor()
# Encrypts the message
encrypted_message = encryptor.update(message) + encryptor.finalize()
# Encrypts the AES key using the receivers public key
encrypted_key = public_key.encrypt(
aes_key,
padding.OAEP(
mgf=padding.MGF1(algorithm=hashes.SHA256()),
algorithm=hashes.SHA256(),
label=None
)
# Bundles to be transmitted easier using pickle
)
payload = pickle.dumps((encrypted_key, iv, encrypted_message))
# Creates a TCP socket and sends the payload
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.connect(("localhost", 65432))
    s.sendall(payload)
    print("Message encrypted and sent successfully.")

Requirement already satisfied: cryptography in c:\users\emelz\appdata\local\packages\pythonsoftwarefoundation.python.3.11_qbz5n2kfra8p0\localcache\local-packages\python311\site-packages (46.0.2)
Requirement already satisfied: cffi>=2.0.0 in c:\users\emelz\appdata\local\packages\pythonsoftwarefoundation.python.3.11_qbz5n2kfra8p0\localcache\local-packages\python311\site-packages (from cryptography) (2.0.0)
Requirement already satisfied: pycparser in c:\users\emelz\appdata\local\packages\pythonsoftwarefoundation.python.3.11_qbz5n2kfra8p0\localcache\local-packages\python311\site-packages (from cffi>=2.0.0->cryptography) (2.23)
✅ Encrypted message sent!

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\emelz\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip

<h2>Step 4: Group Work.pdf</h2>

<h2>(Challenge)Step 5: Digital Signatures</h2>

In [ ]:
#imports libaries and modules
import socket
import pickle
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.exceptions import InvalidSignature
# reads the private key
with open("private_key.pem", "rb") as f:
    private_key = serialization.load_pem_private_key(f.read(), password=None)
# reads the public key
with open("public_key.pem", "rb") as f:
    sender_public_key = serialization.load_pem_public_key(f.read())
#opens a TCP socket connection
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.bind(("localhost", 65432))
    s.listen()

    print("Server is running and waiting for connections...")
# while true, it has 10 seconds so both connections can be recieved
    while True:
        print("\nwaiting for a message... (10s timeout)")
        s.settimeout(10) 
        
        try:
            conn, addr = s.accept()
        except socket.timeout:
            print("receiver timed out.")
            break
# write the data into bytes and assemble it together using the pickle function
        with conn:
            print(f"connected by {addr}")
            data = b""

            while True:
                chunk = conn.recv(4096)
                if not chunk:
                    break
                data += chunk
# here is the pickle function being utilised to put the data together
        encrypted_key, iv, encrypted_message, signature = pickle.loads(data)

        try:
            # public key verifies the information and uses hashing to ensure that the data matches,
            # this is because if the value of the orginial emssage is changed even the slightest, a whole
            # new hash is generated and therefore hashing utilised to ensure the integrity of the data
            sender_public_key.verify(
                signature,
                iv + encrypted_message,
                padding.PSS(
                    mgf=padding.MGF1(hashes.SHA256()),
                    salt_length=padding.PSS.MAX_LENGTH
                ),
                hashes.SHA256()
            )
            print("valid signature")
        except InvalidSignature:
            print("message has been tampered with ")
            continue  
            # if the signature is then valid is then decrypts the aes key
        aes_key = private_key.decrypt(
            encrypted_key,
            padding.OAEP(
                mgf=padding.MGF1(algorithm=hashes.SHA256()),
                algorithm=hashes.SHA256(),
                label=None
            )
        )
#  actual decryption of the message occurs here
        cipher = Cipher(algorithms.AES(aes_key), modes.CFB(iv))
        decryptor = cipher.decryptor()
        message = decryptor.update(encrypted_message) + decryptor.finalize()
# prints the final message
        print(f"decrypted message: {message.decode()}")


running...

waiting for a message... (10s timeout)
connected by ('127.0.0.1', 53162)
message has been tampered with 

waiting for a message... (10s timeout)
connected by ('127.0.0.1', 53164)
valid signature
decrypted message: hello world

waiting for a message... (10s timeout)
receiver timed out.
